# LeetCode 3313 — Find the Last Marked Nodes in Tree bằng Python

- Link: https://leetcode.com/problems/find-the-last-marked-nodes-in-tree/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Lan truyền theo BFS** = khoảng cách; nút bị đánh dấu cuối cùng là **nút xa nhất**; tính chất **đường kính cây**: nút xa nhất của mọi đỉnh là một trong hai đầu mút đường kính

> **Ghi chú Premium:** bài Premium nên LeetCode không trả đề qua API công khai. Đề, ví dụ, ràng buộc và chữ ký `lastMarkedNodes(self, edges)` lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3313/) — vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton (theo doocs):

```python
class Solution:
    def lastMarkedNodes(self, edges: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nhận ra quá trình "mỗi giây đánh dấu mọi nút kề nút đã đánh dấu" chính là **BFS**: nút `v` được đánh dấu tại thời điểm `dist(i, v)`.
2. Quy về: với mỗi `i`, tìm **một** nút xa nhất (độ lệch tâm — eccentricity).
3. Định lý: nếu `(a, b)` là đường kính cây thì với mọi `v`, `max(dist(v, a), dist(v, b)) = ecc(v)` ⇒ chỉ cần **3 lần BFS**.
4. Cách khác: **rerooting DP** (chiều cao xuống + giá trị lên), có lưu luôn nút đạt cực đại.
5. Vì có nhiều đáp án hợp lệ, test bằng **hàm kiểm tra** thay vì so sánh mảng.

## 2. Đề bài & ví dụ

Cây vô hướng `n` nút (`0..n−1`), cạnh `edges`. Ban đầu mọi nút chưa đánh dấu. Mỗi giây, đánh dấu mọi nút chưa đánh dấu có **ít nhất một nút kề đã đánh dấu**. Trả về mảng `nodes` với `nodes[i]` là nút **được đánh dấu sau cùng** nếu tại `t = 0` ta đánh dấu nút `i`. Nếu có nhiều đáp án, trả về bất kỳ.

**Ví dụ 1:** `[[0,1],[0,2]]` → `[2,2,1]` (với `i = 0`, `1` hoặc `2` đều được).
**Ví dụ 2:** `[[0,1]]` → `[1,0]`.
**Ví dụ 3:** `[[0,1],[0,2],[2,3],[2,4]]` → `[3,3,1,1,1]`.

**Ràng buộc (theo doocs):** `2 ≤ n ≤ 10^5`; cạnh tạo thành cây hợp lệ.

In [1]:
from typing import List
from collections import deque

examples = [([[0, 1], [0, 2]], [2, 2, 1]), ([[0, 1]], [1, 0]), ([[0, 1], [0, 2], [2, 3], [2, 4]], [3, 3, 1, 1, 1])]
for edges, exp in examples:
    print(edges, '-> một đáp án hợp lệ:', exp)

[[0, 1], [0, 2]] -> một đáp án hợp lệ: [2, 2, 1]
[[0, 1]] -> một đáp án hợp lệ: [1, 0]
[[0, 1], [0, 2], [2, 3], [2, 4]] -> một đáp án hợp lệ: [3, 3, 1, 1, 1]


## 3. Kiến thức nền

### 3.1. BFS cho khoảng cách trên cây (không đệ quy)

In [2]:
def build(edges):
    n = len(edges) + 1
    g = [[] for _ in range(n)]
    for u, v in edges:
        g[u].append(v); g[v].append(u)
    return g

def bfs(g, src):
    dist = [-1] * len(g)
    dist[src] = 0
    q = deque([src])
    while q:
        u = q.popleft()
        for v in g[u]:
            if dist[v] < 0:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

g = build(examples[2][0])
print('khoảng cách từ 1:', bfs(g, 1))

khoảng cách từ 1: [1, 0, 2, 3, 3]


### 3.2. Đánh dấu lan truyền = BFS

Nút `v` được đánh dấu ở giây `t` ⇔ `dist(i, v) = t` (quy nạp theo `t`: các nút giây `t` là các nút chưa đánh dấu kề một nút giây `t − 1`). Nút cuối cùng là nút có `dist(i, v)` **lớn nhất**.

### 3.3. Đường kính cây

BFS từ một nút bất kỳ tới nút xa nhất `a`; BFS từ `a` tới nút xa nhất `b` ⇒ `dist(a, b)` là đường kính. **Tính chất then chốt:** với mọi nút `v`, nút xa nhất của `v` có thể chọn là `a` hoặc `b`.

Kiểm chứng thực nghiệm tính chất này trên cây ngẫu nhiên (độ lệch tâm của mọi nút bằng `max(dist(v, a), dist(v, b))`):

In [3]:
import random as _r
_g = _r.Random(11)
for _ in range(300):
    n_ = _g.randint(2, 20)
    e_ = [[i, _g.randrange(i)] for i in range(1, n_)]
    g_ = build(e_)
    d0_ = bfs(g_, _g.randrange(n_)); a_ = d0_.index(max(d0_))
    da_ = bfs(g_, a_); b_ = da_.index(max(da_)); db_ = bfs(g_, b_)
    assert all(max(bfs(g_, v)) == max(da_[v], db_[v]) for v in range(n_))
print('Tính chất đường kính đúng trên 300 cây ngẫu nhiên (xuất phát từ nút bất kỳ).')

Tính chất đường kính đúng trên 300 cây ngẫu nhiên (xuất phát từ nút bất kỳ).


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 3: cây `1 – 0 – 2 – {3, 4}`. BFS từ `0`: xa nhất là `3` (khoảng cách 2) ⇒ `a = 3`. BFS từ `3`: xa nhất là `1` (khoảng cách 3) ⇒ `b = 1`. Đường kính `1 – 0 – 2 – 3`.

| `v` | `dist(v, a=3)` | `dist(v, b=1)` | chọn |
|---|---|---|---|
| 0 | 2 | 1 | 3 |
| 1 | 3 | 0 | 3 |
| 2 | 1 | 2 | 1 |
| 3 | 0 | 3 | 1 |
| 4 | 2 | 3 | 1 |

⇒ `[3, 3, 1, 1, 1]` ✓.

## 5. Cách 1 — Brute Force: BFS từ mọi nút

In [4]:
def brute(edges):
    g = build(edges)
    ans = []
    for i in range(len(g)):
        d = bfs(g, i)
        ans.append(d.index(max(d)))
    return ans

def is_valid(edges, ans):
    g = build(edges)
    for i in range(len(g)):
        d = bfs(g, i)
        if d[ans[i]] != max(d):
            return False
    return True

for edges, exp in examples:
    out = brute(edges)
    print(out, '| hợp lệ:', is_valid(edges, out), '| đáp án đề hợp lệ:', is_valid(edges, exp))

[1, 2, 1] | hợp lệ: True | đáp án đề hợp lệ: True
[1, 0] | hợp lệ: True | đáp án đề hợp lệ: True
[3, 3, 1, 1, 1] | hợp lệ: True | đáp án đề hợp lệ: True


**Độ phức tạp:** `O(n²)`. `is_valid` cũng `O(n²)` — chỉ dùng cho test nhỏ; với test lớn ta dùng bộ kiểm tra `O(n)` (mục 12.3).

## 6. Ý tưởng thuật toán / bước đột phá

1. `d0 = bfs(0)`, `a = argmax d0`.
2. `da = bfs(a)`, `b = argmax da`.
3. `db = bfs(b)`.
4. `nodes[v] = a` nếu `da[v] ≥ db[v]`, ngược lại `b`.

Ba lần BFS ⇒ `O(n)`.

**Chứng minh tính chất đường kính (phác thảo):** giả sử `w` là nút xa nhất của `v` nhưng `dist(v, w) > max(dist(v, a), dist(v, b))`. Gọi `p` là nút trên đường `a–b` gần `v` nhất. Đường `v → w` hoặc đi qua `p` hoặc rẽ khỏi đường `a–b` trước đó; trong cả hai trường hợp ta ghép được một đường từ `a` hoặc `b` tới `w` dài hơn `dist(a, b)` — mâu thuẫn với `a–b` là đường kính.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Rerooting DP (chiều cao xuống + lên, kèm nút đạt cực đại)

- `down1[v], node1[v]` = độ sâu lớn nhất trong cây con của `v` và nút đạt được; `down2[v]` = giá trị lớn thứ hai qua **một con khác**.
- `up[v], upnode[v]` = đường dài nhất từ `v` đi **lên** (qua cha).
- `ecc(v) = max(down1[v], up[v])`, nút tương ứng là đáp án.

In [5]:
def rerooting(edges):
    g = build(edges)
    n = len(g)
    parent = [-1] * n
    order = []
    seen = [False] * n; seen[0] = True
    st = [0]
    while st:
        u = st.pop(); order.append(u)
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; st.append(v)
    down1 = [0] * n; node1 = list(range(n)); via1 = [-1] * n
    down2 = [0] * n; node2 = list(range(n))
    for u in reversed(order):                          # hậu thứ tự: con trước
        for v in g[u]:
            if v == parent[u]:
                continue
            cand = down1[v] + 1
            if cand > down1[u]:
                down2[u], node2[u] = down1[u], node1[u]
                down1[u], node1[u], via1[u] = cand, node1[v], v
            elif cand > down2[u]:
                down2[u], node2[u] = cand, node1[v]
    up = [0] * n; upnode = list(range(n))
    for u in order:                                    # tiền thứ tự: cha trước
        for v in g[u]:
            if v == parent[u]:
                continue
            # đường dài nhất từ u không đi vào v
            if via1[u] == v:
                best, bn = down2[u], node2[u]
            else:
                best, bn = down1[u], node1[u]
            if up[u] > best:
                best, bn = up[u], upnode[u]
            up[v], upnode[v] = best + 1, bn
    return [node1[v] if down1[v] >= up[v] else upnode[v] for v in range(n)]

for edges, exp in examples:
    out = rerooting(edges)
    print(out, '| hợp lệ:', is_valid(edges, out))

[1, 2, 1] | hợp lệ: True
[1, 0] | hợp lệ: True
[3, 3, 1, 1, 1] | hợp lệ: True


Khi `u` không có con nào khác `v`, `down2[u] = 0` và `node2[u] = u` — tức "đường" từ `u` dừng ngay tại `u`, đúng.

### 7.2. Cách 3 — Ba lần BFS theo đường kính (lời giải chính)

In [6]:
def diameter_solution(edges):
    g = build(edges)
    d0 = bfs(g, 0)
    a = d0.index(max(d0))
    da = bfs(g, a)
    b = da.index(max(da))
    db = bfs(g, b)
    return [a if x >= y else b for x, y in zip(da, db)]

for edges, exp in examples:
    out = diameter_solution(edges)
    print(out, '| hợp lệ:', is_valid(edges, out))

[1, 2, 1] | hợp lệ: True
[1, 0] | hợp lệ: True
[3, 3, 1, 1, 1] | hợp lệ: True


## 8. TRACE từng bước

Trace cách 3 trên một cây 8 nút.

In [7]:
def trace(edges):
    g = build(edges)
    d0 = bfs(g, 0); a = d0.index(max(d0))
    da = bfs(g, a); b = da.index(max(da))
    db = bfs(g, b)
    print('BFS từ 0:', d0, '-> a =', a)
    print(f'BFS từ a={a}:', da, '-> b =', b, '| đường kính =', da[b])
    print(f'BFS từ b={b}:', db)
    ans = [a if x >= y else b for x, y in zip(da, db)]
    print('đáp án:', ans, '| hợp lệ:', is_valid(edges, ans))

trace([[0, 1], [1, 2], [1, 3], [3, 4], [0, 5], [5, 6], [6, 7]])

BFS từ 0: [0, 1, 2, 2, 3, 1, 2, 3] -> a = 4
BFS từ a=4: [3, 2, 3, 1, 0, 4, 5, 6] -> b = 7 | đường kính = 6
BFS từ b=7: [3, 4, 5, 5, 6, 2, 1, 0]
đáp án: [4, 7, 7, 7, 7, 4, 4, 4] | hợp lệ: True


## 9. Code nộp LeetCode

(Chữ ký theo doocs — xem ghi chú Premium. Dùng BFS lặp thay vì DFS đệ quy như mã tham khảo để tránh tràn đệ quy với cây dạng đường thẳng.)

In [8]:
class Solution:
    def lastMarkedNodes(self, edges: List[List[int]]) -> List[int]:
        n = len(edges) + 1
        g = [[] for _ in range(n)]
        for u, v in edges:
            g[u].append(v)
            g[v].append(u)

        def bfs(src):
            dist = [-1] * n
            dist[src] = 0
            q = [src]
            for u in q:                     # danh sách đóng vai hàng đợi
                du = dist[u] + 1
                for v in g[u]:
                    if dist[v] < 0:
                        dist[v] = du
                        q.append(v)
            return dist, q[-1]              # nút thăm cuối cùng là một nút xa nhất

        _, a = bfs(0)
        da, b = bfs(a)
        db, _ = bfs(b)
        return [a if x >= y else b for x, y in zip(da, db)]

for edges, exp in examples:
    out = Solution().lastMarkedNodes(edges)
    print(out, '| hợp lệ:', is_valid(edges, out))

[2, 2, 1] | hợp lệ: True
[1, 0] | hợp lệ: True
[4, 4, 1, 1, 1] | hợp lệ: True


**Mẹo:** trong BFS, nút được thăm **cuối cùng** có khoảng cách lớn nhất — không cần `index(max(...))`.

## 10. Vì sao thuật toán đúng

1. **Thời điểm đánh dấu = khoảng cách** (mục 3.2).
2. **`a` là một đầu mút đường kính:** BFS từ nút bất kỳ, nút xa nhất là đầu mút của một đường kính (bổ đề kinh điển, chứng minh bằng phản chứng tương tự mục 6).
3. **`b` xa nhất từ `a`** ⇒ `(a, b)` là đường kính.
4. **Nút xa nhất của mọi `v` thuộc `{a, b}`** (mục 6) ⇒ chọn cái có khoảng cách lớn hơn là một đáp án hợp lệ.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS từ mọi nút | `O(n²)` | `O(n)` |
| 2. Rerooting DP | `O(n)` | `O(n)` |
| 3. Ba lần BFS (đường kính) | `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên (kiểm bằng hàm hợp lệ)

In [9]:
impls = {'rerooting': rerooting, 'diameter': diameter_solution, 'submit': lambda e: Solution().lastMarkedNodes(e)}
edge_cases = [
    [[0, 1], [1, 2], [2, 3]],                 # đường thẳng
    [[0, 1], [0, 2], [0, 3], [0, 4]],         # hình sao
    [[1, 0]],                                 # thứ tự đầu mút ngược
]
for edges in [e for e, _ in examples] + edge_cases:
    for name, f in impls.items():
        assert is_valid(edges, f(edges)), (name, edges)
assert Solution().lastMarkedNodes([[0, 1], [1, 2], [2, 3]]) == [3, 3, 0, 0]
print('Ví dụ đề và ca biên: mọi đáp án hợp lệ.')

Ví dụ đề và ca biên: mọi đáp án hợp lệ.


### 12.2. Đối chiếu ngẫu nhiên (độ lệch tâm phải khớp BFS từ mọi nút)

In [10]:
import random
rng = random.Random(3313)
def random_tree(n):
    nodes = list(range(n)); rng.shuffle(nodes)
    return [[nodes[i], nodes[rng.randrange(i)]] for i in range(1, n)]

cnt = 0
for _ in range(400):
    edges = random_tree(rng.randint(2, 30))
    for name, f in impls.items():
        assert is_valid(edges, f(edges)), (name, edges)
    cnt += 1
print('Mọi cách cho đáp án hợp lệ trên', cnt, 'cây ngẫu nhiên.')

Mọi cách cho đáp án hợp lệ trên 400 cây ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5` (kiểm hợp lệ bằng độ lệch tâm từ rerooting)

Kiểm tra toàn bộ `O(n²)` là quá chậm, nên ta lấy 30 nút ngẫu nhiên, chạy BFS đầy đủ từ mỗi nút và kiểm tra: khoảng cách tới đáp án của lời giải nộp, tới đáp án của **rerooting** (cách tính độc lập), và `max(da[v], db[v])` đều bằng độ lệch tâm thật.

In [11]:
import time
N = 10 ** 5
big = {
    'cây ngẫu nhiên': random_tree(N),
    'đường thẳng': [[i, i + 1] for i in range(N - 1)],
    'hình sao': [[0, i] for i in range(1, N)],
    'sâu bướm (caterpillar)': [[i, i + 1] for i in range(N // 2 - 1)] + [[i % (N // 2), N // 2 + i] for i in range(N - N // 2)],
}
for name, edges in big.items():
    t = time.time(); ans = Solution().lastMarkedNodes(edges); t1 = time.time() - t
    t = time.time(); ref = rerooting(edges); t2 = time.time() - t
    g = build(edges)
    d0 = bfs(g, 0); a = d0.index(max(d0)); da = bfs(g, a); b = da.index(max(da)); db = bfs(g, b)
    ecc = [max(x, y) for x, y in zip(da, db)]
    sample = rng.sample(range(N), 30)
    for v in sample:
        dv = bfs(g, v)
        assert dv[ans[v]] == max(dv) == dv[ref[v]] == ecc[v]
    print(f"{name:24s}: nộp {t1:.2f}s | rerooting {t2:.2f}s | 30 nút ngẫu nhiên hợp lệ")

cây ngẫu nhiên          : nộp 0.26s | rerooting 0.25s | 30 nút ngẫu nhiên hợp lệ


đường thẳng             : nộp 0.15s | rerooting 0.10s | 30 nút ngẫu nhiên hợp lệ


hình sao                : nộp 0.07s | rerooting 0.10s | 30 nút ngẫu nhiên hợp lệ


sâu bướm (caterpillar)  : nộp 0.10s | rerooting 0.15s | 30 nút ngẫu nhiên hợp lệ


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ dùng **một** đầu mút đường kính

In [12]:
def wrong_one_endpoint(edges):
    g = build(edges)
    d0 = bfs(g, 0); a = d0.index(max(d0))
    return [a] * len(g)                                  # SAI: với chính a (và các nút gần a), a là nút gần nhất!

edges = examples[2][0]
out = wrong_one_endpoint(edges)
print('sai :', out, '| hợp lệ:', is_valid(edges, out), '| đúng:', Solution().lastMarkedNodes(edges))

sai : [3, 3, 3, 3, 3] | hợp lệ: False | đúng: [4, 4, 1, 1, 1]


**Vì sao sai:** với `v = a`, khoảng cách tới `a` là 0. Phải so sánh hai đầu mút.

### Lỗi 2 — Lấy cả hai "đầu mút" bằng BFS từ cùng nút 0

In [13]:
def wrong_both_from_zero(edges):
    g = build(edges)
    d0 = bfs(g, 0)
    order = sorted(range(len(g)), key=lambda v: -d0[v])
    a, b = order[0], order[1]                            # SAI: b không phải đầu mút đường kính
    da, db = bfs(g, a), bfs(g, b)
    return [a if x >= y else b for x, y in zip(da, db)]

edges = [[1, 0], [2, 1]]                                   # đường thẳng 0 - 1 - 2
out = wrong_both_from_zero(edges)
print('sai :', out, '| hợp lệ:', is_valid(edges, out), '| đúng:', Solution().lastMarkedNodes(edges))

sai : [2, 2, 1] | hợp lệ: False | đúng: [2, 2, 0]


**Vì sao sai:** trên đường `0 – 1 – 2`, hai nút xa nhất từ `0` là `2` và `1`; nhưng `1` không phải đầu mút đường kính. Với `v = 2`, cả hai "ứng viên" đều gần (`0` và `1`), trong khi nút xa nhất thật là `0`. Đầu mút thứ hai phải tìm bằng BFS **từ `a`**.

### Lỗi 3 — Chọn ngược (`a` khi `da < db`)

In [14]:
def wrong_flip(edges):
    g = build(edges)
    d0 = bfs(g, 0); a = d0.index(max(d0))
    da = bfs(g, a); b = da.index(max(da)); db = bfs(g, b)
    return [a if x < y else b for x, y in zip(da, db)]   # SAI: chọn đầu mút gần hơn

edges = examples[2][0]
out = wrong_flip(edges)
print('sai :', out, '| hợp lệ:', is_valid(edges, out))

sai : [1, 1, 3, 3, 3] | hợp lệ: False


**Vì sao sai:** cần đầu mút **xa** hơn (thời điểm đánh dấu muộn hơn).

### Lỗi 4 — DFS đệ quy (như mã tham khảo) trên cây đường thẳng

In [15]:
import sys
def recursive_dist(g, src):
    dist = [-1] * len(g); dist[src] = 0
    def dfs(u, p):
        for v in g[u]:
            if v != p:
                dist[v] = dist[u] + 1
                dfs(v, u)
    dfs(src, -1)
    return dist

g_line = build([[i, i + 1] for i in range(N - 1)])
try:
    recursive_dist(g_line, 0)
    print('không lỗi')
except RecursionError:
    print('RecursionError với cây đường thẳng 1e5 nút -> dùng BFS/DFS lặp')

RecursionError với cây đường thẳng 1e5 nút -> dùng BFS/DFS lặp


**Vì sao sai:** giới hạn đệ quy mặc định của Python khoảng 1000 tầng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lan truyền mỗi giây sang hàng xóm" ⇒ BFS ⇒ khoảng cách.
- "Nút xa nhất / độ lệch tâm của **mọi** nút trên cây" ⇒ hai đầu mút đường kính (hoặc rerooting).
- Bài cho phép nhiều đáp án ⇒ viết **hàm kiểm tra** để test.

## 15. Biến thể của bài

1. Trả về **thời điểm** nút cuối được đánh dấu ⇒ `ecc(i) = max(da[i], db[i])`.
2. Đánh dấu ban đầu **nhiều** nút ⇒ BFS đa nguồn, nút cuối là nút xa nhất tới tập nguồn.
3. Cạnh có **trọng số** (thời gian lan khác nhau) ⇒ vẫn đúng tính chất đường kính với trọng số không âm (dùng DFS/BFS tính tổng trọng số trên cây).
4. Tâm cây (nút có `ecc` nhỏ nhất) — LC 310 Minimum Height Trees.

In [16]:
# Biến thể 1, 2, 4
def last_times(edges):
    g = build(edges)
    d0 = bfs(g, 0); a = d0.index(max(d0)); da = bfs(g, a); b = da.index(max(da)); db = bfs(g, b)
    return [max(x, y) for x, y in zip(da, db)]

def multi_source_last(edges, sources):
    g = build(edges)
    dist = [-1] * len(g); q = deque()
    for s in sources:
        dist[s] = 0; q.append(s)
    while q:
        u = q.popleft()
        for v in g[u]:
            if dist[v] < 0:
                dist[v] = dist[u] + 1; q.append(v)
    return dist.index(max(dist)), max(dist)

def centers(edges):
    ecc = last_times(edges)
    m = min(ecc)
    return sorted(v for v in range(len(ecc)) if ecc[v] == m)

rng = random.Random(15)
for _ in range(200):
    edges = random_tree(rng.randint(2, 15))
    g = build(edges)
    assert last_times(edges) == [max(bfs(g, i)) for i in range(len(g))]
    src = rng.sample(range(len(g)), min(len(g), rng.randint(1, 3)))
    node, t = multi_source_last(edges, src)
    assert t == max(min(bfs(g, s)[v] for s in src) for v in range(len(g)))
    assert len(centers(edges)) in (1, 2)
print('Biến thể 1, 2 khớp vét cạn; tâm cây có 1 hoặc 2 nút. Ví dụ tâm của ví dụ 3:', centers(examples[2][0]))

Biến thể 1, 2 khớp vét cạn; tâm cây có 1 hoặc 2 nút. Ví dụ tâm của ví dụ 3: [0, 2]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | BFS từ mọi nút | `O(n²)` | Oracle |
| 2 | Rerooting DP | `O(n)` | Tổng quát (mở rộng được nhiều bài) |
| 3 | Ba BFS theo đường kính | `O(n)` | **Nộp bài**, ngắn gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `edges = [[0, 1], [1, 2]]`.

**Bài 2 (tính tay):** thời điểm nút cuối được đánh dấu với mỗi `i` cho cây ví dụ 3.

**Bài 3 (điền TODO):** trả về đáp án với quy tắc hòa: chọn nút có **số hiệu nhỏ nhất** trong các nút xa nhất.

**Bài 4 (tự viết lại):** biến thể 3 — cây có trọng số cạnh dương.

**Bài 5 (mini challenge):** với mỗi `i`, đếm **số** nút được đánh dấu sau cùng (cùng thời điểm `ecc(i)`).

In [17]:
def smallest_tiebreak_todo(edges):
    # TODO
    raise NotImplementedError

def weighted_last_todo(n, wedges):
    # TODO
    raise NotImplementedError

def count_last_todo(edges):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đường thẳng `0 – 1 – 2`. (2) Nút 1 cách đều 0 và 2. (3) `[2, 0 hoặc 2, 0]`.

**Bài 2.** (1) `max(da, db)`. (2) `da = [2,3,1,0,2]`, `db = [1,0,2,3,3]`. (3) `[2, 3, 2, 3, 3]`.

**Bài 3.** (1) Cần **tất cả** nút xa nhất ⇒ không chỉ hai đầu mút. (2) Nút xa nhất của `v` luôn là một đầu mút của **một** đường kính nào đó; rerooting với so sánh `(độ sâu, −số hiệu)`. (3) Vét cạn nhỏ.

**Bài 4.** (1) Thay BFS bằng duyệt cây cộng trọng số. (2) Tính chất đường kính vẫn đúng với trọng số dương. (3) Vét cạn bằng Dijkstra/duyệt từ mọi nút.

**Bài 5.** (1) Rerooting đếm `(độ sâu lớn nhất, số nút đạt)` cho cả chiều xuống và lên. (2) Gộp hai chiều khi bằng nhau. (3) Vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
out = Solution().lastMarkedNodes([[0, 1], [1, 2]])
assert out[0] == 2 and out[2] == 0 and out[1] in (0, 2)
assert last_times(examples[2][0]) == [2, 3, 2, 3, 3]

def smallest_tiebreak(edges):
    g = build(edges); n = len(g)
    parent = [-1] * n; order = []; seen = [False] * n; seen[0] = True; st = [0]
    while st:
        u = st.pop(); order.append(u)
        for v in g[u]:
            if not seen[v]:
                seen[v] = True; parent[v] = u; st.append(v)
    best = lambda p, q: p if (p[0], -p[1]) >= (q[0], -q[1]) else q    # (độ dài, nút)
    down = [(0, v) for v in range(n)]; top2 = [[(0, v), (0, v)] for v in range(n)]; via = [-1] * n
    for u in reversed(order):
        for v in g[u]:
            if v == parent[u]:
                continue
            cand = (down[v][0] + 1, down[v][1])
            a1, a2 = top2[u]
            if (cand[0], -cand[1]) > (a1[0], -a1[1]):
                top2[u] = [cand, a1]; via[u] = v
            elif (cand[0], -cand[1]) > (a2[0], -a2[1]):
                top2[u] = [a1, cand]
        down[u] = top2[u][0]
    up = [(-1, n)] * n; up[0] = (0, 0)
    for u in order:
        for v in g[u]:
            if v == parent[u]:
                continue
            side = top2[u][1] if via[u] == v else top2[u][0]
            b = best(side, up[u])
            up[v] = (b[0] + 1, b[1])
    return [best(down[v], up[v])[1] for v in range(n)]

rng = random.Random(3)
for _ in range(200):
    edges = random_tree(rng.randint(2, 14)); g = build(edges)
    exp = []
    for i in range(len(g)):
        d = bfs(g, i); m = max(d); exp.append(min(v for v in range(len(g)) if d[v] == m))
    assert smallest_tiebreak(edges) == exp, edges
print('Bài 1, 2 đúng | Bài 3 khớp vét cạn.')

Bài 1, 2 đúng | Bài 3 khớp vét cạn.


In [19]:
# Bài 4, 5
def weighted_last(n, wedges):
    g = [[] for _ in range(n)]
    for u, v, w in wedges:
        g[u].append((v, w)); g[v].append((u, w))
    def far(src):
        dist = [-1] * n; dist[src] = 0; st = [src]
        while st:
            u = st.pop()
            for v, w in g[u]:
                if dist[v] < 0:
                    dist[v] = dist[u] + w; st.append(v)
        return dist
    d0 = far(0); a = d0.index(max(d0)); da = far(a); b = da.index(max(da)); db = far(b)
    return [a if x >= y else b for x, y in zip(da, db)], far

def count_last(edges):
    g = build(edges)
    res = []
    for i in range(len(g)):
        d = bfs(g, i); m = max(d); res.append(d.count(m))
    return res

rng = random.Random(4)
for _ in range(200):
    n = rng.randint(2, 14); edges = random_tree(n)
    wedges = [[u, v, rng.randint(1, 9)] for u, v in edges]
    ans, far = weighted_last(n, wedges)
    for i in range(n):
        d = far(i); assert d[ans[i]] == max(d)
print('Bài 4 hợp lệ trên cây có trọng số. Bài 5 ví dụ 3:', count_last(examples[2][0]))

Bài 4 hợp lệ trên cây có trọng số. Bài 5 ví dụ 3: [2, 2, 1, 1, 1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
_, a = bfs(0)        # nút thăm cuối = xa nhất
da, b = bfs(a)
db, _ = bfs(b)
return [a if da[v] >= db[v] else b for v in range(n)]
```

**Checklist:**

- [ ] Thời điểm đánh dấu = khoảng cách BFS?
- [ ] `b` tìm bằng BFS **từ `a`** (không phải từ 0)?
- [ ] So sánh hai đầu mút, chọn cái **xa** hơn?
- [ ] BFS/DFS lặp (cây đường thẳng `10^5`)?
- [ ] Test bằng hàm kiểm tra vì có nhiều đáp án?